<centre>$$ Multi-Head Attention: $$</centre>

In [ ]:
import torch 
import torch.nn as nn
from torch.nn import functional as F
import math 
class multi_head_attention(nn.Module):
    def __init__(self,d_model,num_heads):
        super().__init__()
        assert d_model%num_heads==0 #"d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model//num_heads
        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model)
        self.W_o = nn.Linear(d_model,d_model)
    def split_heads(self,x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)
    def forward(self,Q,K,V , mask_size=None):
        batch_size = Q.size(0)
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)

        Q = self.split_heads(Q, batch_size)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)
        scores = torch.matmul(Q,K.transpose(-2,-1))
        scores = scores / math.sqrt(self.d_k)
        weigths = F.softmax(scores,dim=-1)
        output = torch.matmul(weigths,V)
        output = output.transpose(1,2)
        output = output.contiguous().view(batch_size,-1,self.d_model)
        output = self.W_o(output)
        return output

d_model = 512
num_heads = 8 
seq_length = 10
batch_size = 2
aha = multi_head_attention(d_model,num_heads)
x = torch.rand(batch_size,seq_length,d_model)
output = aha(x,x,x)
print(output.shape)
print(f"output: {output.shape}")
print(f"weights: {weigths.shape}")



